# Lab 3: Google Earth Engine & Sentinel-2 Acquisition (2 Hours)


## ⏱️ Time Allocation
- **Part 1 (30 min):** GEE setup and authentication
- **Part 2 (55 min):** Data discovery - AOI, querying, visualization
- **Part 3 (35 min):** Export full tiles + CORINE labels
- **Part 4 (10+ min):** Advanced topics (optional)


## 🎯 Learning Objectives


### Core (Essential - Everyone Should Complete)
- ✅ Authenticate with Google Earth Engine
- ✅ Choose your own AOI (keep it within CORINE coverage: Europe + Iceland)
- ✅ Identify the Sentinel-2 MGRS tile that contains the AOI
- ✅ Download **4 full acquisitions** (complete tile footprint) for that tile
- ✅ Download and align the matching **CORINE land cover labels** for the tile
- ✅ Save metadata so the imagery and labels can be used for fine-tuning


### Optional (For Early Finishers)
- 🔵 Calculate spectral indices (NDVI, NDWI, NBR)
- 🔵 Advanced cloud masking with QA bands
- 🔵 Temporal analysis and time series
- 🔵 Multiple AOI comparison
- 🔵 Custom band combinations


### ⚠️ Pre-Lab Requirement
**You MUST have a Google Earth Engine account approved before this lab!**
- Apply at: https://earthengine.google.com/signup
- Approval takes 1-2 days
- Use your university or personal Gmail account


---


## Section 1: Introduction to Google Earth Engine (10 min)


### What is Google Earth Engine (GEE)?
Google Earth Engine is a cloud-based platform for planetary-scale geospatial analysis:
- **Petabyte-scale** catalog of satellite imagery
- **Server-side processing** - no downloads needed for analysis
- **Free** for research and education
- **Python and JavaScript APIs**


### Why GEE for This Course?
- ✅ Easy access to Sentinel-2 archive
- ✅ Built-in cloud filtering
- ✅ Fast querying and visualization
- ✅ No storage needed until export


### GEE Data Catalog
- Sentinel-2 (10m optical)
- Landsat 5/7/8/9 (30m optical)
- MODIS (250m-1km)
- Sentinel-1 (10m SAR)
- And many more!


### Sentinel-2 in GEE
- **Collection:** `COPERNICUS/S2_SR` (Surface Reflectance)
- **Bands:** 13 spectral bands (10m, 20m, 60m resolution)
- **Coverage:** Global, since 2015
- **Revisit:** 5 days (with both satellites)

## Section 2: GEE Setup and Authentication (5 min)

### Step 1: Install Earth Engine API (In case not installed already)

In [ ]:
# Install required packages (run once on terminal)
source /p/project1/training2600/$USER/envs/ml_eo_course/bin/activate
pip install --user earthengine-api geemap folium geopandas

### Step 2: Authenticate GEE
You'll need to authenticate once per environment:

In [32]:
import ee

# Trigger authentication (will open browser)
try:
    ee.Initialize()
    print("✅ Already authenticated!")
except:
    print("🔐 Authentication required...")
    ee.Authenticate()
    ee.Initialize()
    print("✅ Authentication successful!")

✅ Already authenticated!


**Authentication Steps:**
1. Click the link that appears
2. Sign in with your Google account
3. Grant Earth Engine permissions
4. Copy the authorization code
5. Paste it back into the notebook

In [ ]:
# Import additional libraries
import geemap
import folium
import datetime
import pandas as pd
from pathlib import Path
import json
import os


print("✅ All libraries loaded successfully!")

✅ All libraries loaded successfully!


## Section 3: Define Area of Interest (AOI) (8 min)


### Choose Your Study Area
Pick an AOI you care about **within CORINE coverage** (Europe + Iceland). The CORINE labels we will download later are only available there. Example ideas:


- **Reykjavik Region:** Urban + coastal
- **Þingvellir:** Vegetation + volcanic
- **Vatnajökull:** Glaciers + bare rock


You'll draw your AOI below. A small polygon is fine—the download will cover the **entire Sentinel-2 tile** containing that AOI.

In [ ]:
# Fallback AOI (edit if you prefer typing a bounding box)
# [west, south, east, north] near Þingvellir
fallback_aoi_coords = [-22.5, 63.8, -18.5, 64.8]


# Create fallback geometry (will be replaced if you draw your own)
aoi = ee.Geometry.Rectangle(fallback_aoi_coords)


print("Using fallback AOI for now — draw your own below to replace it.")
print(f"AOI Center: {aoi.centroid().coordinates().getInfo()}")
print(f"AOI Area: {aoi.area().divide(1e6).getInfo():.2f} km²")

AOI Center: [-21.14999999999969, 64.29995584118119]
AOI Area: 321.71 km²


In [ ]:
# Visualize AOI on interactive map
Map = geemap.Map(center=[64.3, -20.5], zoom=8)
Map.addLayer(aoi, {'color': 'red'}, 'AOI')
Map.add_basemap('SATELLITE')
Map

Map(center=[64.3, -21.15], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

### Alternative: Draw Your Own AOI
You can also draw directly on the map:

In [ ]:
# Interactive AOI drawing
Map = geemap.Map(center=[64.3, -21.15], zoom=10)
Map.add_basemap('SATELLITE')


# Instructions:
print("📍 Use the drawing tools to define your AOI:")
print("   1. Click the rectangle/polygon tool on the left")
print("   2. Draw your area of interest (keep it small; must be inside CORINE coverage)")
print("   3. Run the next cell to lock it in and extract coordinates")


Map

In [ ]:
# Extract drawn AOI (if you used drawing tools)
# aoi = Map.draw_last_feature.geometry()
# print(f"Custom AOI coordinates: {aoi.bounds().getInfo()['coordinates']}")

### Lock in the AOI
Run the cell below after drawing. It will use your drawn geometry if present; otherwise it keeps the fallback bounding box.

In [ ]:
# Use drawn geometry if available, otherwise keep the fallback rectangle
try:
    drawn_feature = Map.user_roi
except Exception:
    drawn_feature = None


if drawn_feature:
    aoi = drawn_feature.geometry()
    chosen_source = "user-drawn AOI"
else:
    aoi = ee.Geometry.Rectangle(fallback_aoi_coords)
    chosen_source = "fallback bounding box"


aoi_bounds = aoi.bounds()
aoi_coords = aoi_bounds.coordinates().getInfo()
aoi_area_km2 = aoi.area().divide(1e6).getInfo()


print(f"Using {chosen_source} for all downstream steps.")
print(f"AOI area: {aoi_area_km2:.2f} km²")
print(f"AOI bounds: {aoi_coords}")

## Section 4: Query Sentinel-2 Image Collection (12 min)

### Define Search Parameters

In [56]:
# Date range: Summer 2024 (less snow, more vegetation)
start_date = '2024-06-01'
end_date = '2024-09-30'

# Cloud cover threshold
max_cloud_cover = 20  # percent

print(f"🔍 Searching for Sentinel-2 scenes:")
print(f"   Date Range: {start_date} to {end_date}")
print(f"   Max Cloud Cover: {max_cloud_cover}%")
print(f"   AOI: {aoi.area().divide(1e6).getInfo():.2f} km²")

🔍 Searching for Sentinel-2 scenes:
   Date Range: 2024-06-01 to 2024-09-30
   Max Cloud Cover: 20%
   AOI: 321.71 km²


### Query Image Collection

In [57]:
# Query Sentinel-2 Surface Reflectance (Level 2A)
collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
              .filterBounds(aoi)
              .filterDate(start_date, end_date)
              .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', max_cloud_cover)))

# Get collection size
count = collection.size().getInfo()
print(f"\n✅ Found {count} scenes matching criteria")


✅ Found 19 scenes matching criteria


### Identify the Sentinel-2 tile covering your AOI
We want the **full MGRS tile** that contains (or mostly overlaps) the AOI. We will later export four acquisitions of this tile and the matching CORINE land cover labels.

In [ ]:
# Find the dominant MGRS tile that intersects the AOI
tile_hist = collection.aggregate_histogram('MGRS_TILE').getInfo()
tile_series = pd.Series(tile_hist).sort_values(ascending=False)


mgrs_tile = tile_series.index[0]
mgrs_tile_hits = int(tile_series.iloc[0])


s2_grid = ee.FeatureCollection('COPERNICUS/S2_GRID_MGRS')
tile_feature = s2_grid.filter(ee.Filter.eq('MGRS_TILE', mgrs_tile)).first()
tile_geom = tile_feature.geometry()
tile_area_km2 = tile_geom.area().divide(1e6).getInfo()


# Restrict collection to this tile
tile_collection = collection.filter(ee.Filter.eq('MGRS_TILE', mgrs_tile))
tile_collection_size = tile_collection.size().getInfo()


print(f"Chosen MGRS tile: {mgrs_tile}")
print(f"Images overlapping AOI (all tiles): {count}")
print(f"Images in chosen tile: {tile_collection_size}")
print(f"Tile area: {tile_area_km2:.1f} km²")
print(f"AOI is within this tile: {tile_geom.contains(aoi).getInfo()}")

In [ ]:
# Visualize AOI and selected MGRS tile
tile_center = tile_geom.centroid().coordinates().getInfo()
Map = geemap.Map(center=[tile_center[1], tile_center[0]], zoom=9)
Map.add_basemap('SATELLITE')
Map.addLayer(tile_geom, {'color': 'blue'}, f'MGRS Tile {mgrs_tile}')
Map.addLayer(aoi, {'color': 'red'}, 'AOI', opacity=0.4)
Map

### Inspect Scene Metadata

In [ ]:
# Extract metadata for scenes inside the selected tile
def extract_metadata(image):
    return ee.Feature(None, {
        'scene_id': image.get('PRODUCT_ID'),
        'date': image.date().format('YYYY-MM-dd'),
        'cloud_cover': image.get('CLOUDY_PIXEL_PERCENTAGE'),
        'solar_azimuth': image.get('MEAN_SOLAR_AZIMUTH_ANGLE'),
        'solar_zenith': image.get('MEAN_SOLAR_ZENITH_ANGLE'),
        'mgrs_tile': image.get('MGRS_TILE')
    })


metadata = tile_collection.map(extract_metadata).getInfo()['features']
df_scenes = pd.DataFrame([f['properties'] for f in metadata])


print("\n📊 Available Scenes in Selected Tile:")
print(df_scenes.to_string(index=False))


📊 Available Scenes:
 cloud_cover       date                                                     scene_id  solar_azimuth  solar_zenith
   17.767625 2024-06-09 S2B_MSIL2A_20240609T125309_N0510_R138_T27WWM_20240609T135552     171.377886     41.632274
    6.493246 2024-06-10 S2A_MSIL2A_20240610T131301_N0510_R081_T27WWM_20240610T155847     178.237790     41.363926
   12.945187 2024-06-10 S2A_MSIL2A_20240610T131301_N0510_R081_T27WWM_20240610T172147     178.237790     41.363926
    0.057001 2024-06-29 S2B_MSIL2A_20240629T125309_N0510_R138_T27WVM_20240629T135617     167.064513     41.703204
    6.658737 2024-06-29 S2B_MSIL2A_20240629T125309_N0510_R138_T27WWM_20240629T135617     169.902172     41.524312
    1.300875 2024-07-07 S2A_MSIL2A_20240707T130301_N0510_R038_T27WWM_20240707T155249     172.913902     42.074995
    3.306202 2024-08-26 S2A_MSIL2A_20240826T130301_N0511_R038_T27WVM_20240826T154947     172.297391     54.495512
    6.075167 2024-08-26 S2A_MSIL2A_20240826T130301_N0511_R038_T27WW

In [59]:
# Sort by cloud cover and select best scenes
df_scenes_sorted = df_scenes.sort_values('cloud_cover')
print("\n🌤️ Top 5 Clearest Scenes:")
print(df_scenes_sorted.head().to_string(index=False))


🌤️ Top 5 Clearest Scenes:
 cloud_cover       date                                                     scene_id  solar_azimuth  solar_zenith
    0.057001 2024-06-29 S2B_MSIL2A_20240629T125309_N0510_R138_T27WVM_20240629T135617     167.064513     41.703204
    0.432559 2024-09-13 S2B_MSIL2A_20240913T131259_N0511_R081_T27WWM_20240913T152106     179.630060     60.958178
    0.846198 2024-09-15 S2A_MSIL2A_20240915T130301_N0511_R038_T27WVM_20240915T155047     174.674799     61.819436
    1.300875 2024-07-07 S2A_MSIL2A_20240707T130301_N0510_R038_T27WWM_20240707T155249     172.913902     42.074995
    2.586500 2024-09-23 S2B_MSIL2A_20240923T131259_N0511_R081_T27WWM_20240923T165937     180.615841     64.829451


### Visualize Best Scene

In [ ]:
# Get the clearest image within the selected tile
best_image = ee.Image(tile_collection.sort('CLOUDY_PIXEL_PERCENTAGE').first())


# Visualization parameters for True Color (RGB)
vis_params_rgb = {
    'min': 0,
    'max': 3000,
    'bands': ['B4', 'B3', 'B2'],  # Red, Green, Blue
    'gamma': 1.4
}


# Visualization for False Color (NIR, Red, Green) - highlights vegetation
vis_params_nir = {
    'min': 0,
    'max': 3000,
    'bands': ['B8', 'B4', 'B3'],  # NIR, Red, Green
    'gamma': 1.4
}


tile_center = tile_geom.centroid().coordinates().getInfo()


# Create map with both visualizations
Map = geemap.Map(center=[tile_center[1], tile_center[0]], zoom=10)
Map.addLayer(best_image, vis_params_rgb, 'True Color (RGB)')
Map.addLayer(best_image, vis_params_nir, 'False Color (NIR)', shown=False)
Map.addLayer(tile_geom, {'color': 'blue'}, f'Tile {mgrs_tile}', opacity=0.25)
Map.addLayer(aoi, {'color': 'red'}, 'AOI', opacity=0.4)


# Add scene info
scene_date = best_image.date().format('YYYY-MM-dd').getInfo()
cloud_pct = best_image.get('CLOUDY_PIXEL_PERCENTAGE').getInfo()
print(f"\n🖼️ Displaying Best Scene in Tile {mgrs_tile}:")
print(f"   Date: {scene_date}")
print(f"   Cloud Cover: {cloud_pct:.1f}%")


Map

## Section 5: Select and Export Tiles (10 min)

### Select 4 Best Scenes for Training
We'll select 4 complete Sentinel-2 tiles with minimal cloud cover.
These tiles will be patchified (divided into 224×224 patches) in Lab 4:

In [ ]:
# Select up to 4 scenes across the season within the chosen tile
requested_scenes = 4
available_scenes = tile_collection.size().getInfo()
num_scenes = min(requested_scenes, available_scenes)


if available_scenes < requested_scenes:
    print(f"⚠️ Only {available_scenes} scenes available in tile {mgrs_tile}; selecting all of them.")


selected_collection = tile_collection.sort('CLOUDY_PIXEL_PERCENTAGE').limit(num_scenes)


# Get dates of selected scenes
selected_dates = selected_collection.aggregate_array('system:time_start').getInfo()
selected_dates = [datetime.datetime.fromtimestamp(d/1000).strftime('%Y-%m-%d') for d in selected_dates]


print(f"\n✅ Selected {num_scenes} scenes for dataset in tile {mgrs_tile}:")
for i, date in enumerate(selected_dates, 1):
    print(f"   {i}. {date}")


✅ Selected 4 scenes for dataset:
   1. 2024-06-29
   2. 2024-09-13
   3. 2024-09-15
   4. 2024-07-07


### Prepare Tiles for Export
We'll select the best cloud-free tiles at full resolution:

In [ ]:
# Select all bands for full tiles (will select specific bands later if needed)
# We keep all bands so students can choose their preferred band combinations in Lab 4

# Cloud masking function (optional - can improve results)
def mask_clouds(image):
    """Mask clouds using SCL band (Scene Classification Layer)"""
    scl = image.select('SCL')
    # Keep clear (4,5,6) and exclude clouds (8,9,10)
    mask = scl.neq(8).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(3))
    return image.updateMask(mask)

# Apply cloud masking
masked_collection = selected_collection.map(mask_clouds)

print("✅ Applied cloud masking to selected tiles")
print(f"   Tiles will be exported at full 10m resolution")
print(f"   Bands available: B2, B3, B4, B5, B6, B7, B8, B8A, B11, B12")

✅ Applied cloud masking to selected scenes


### Get CORINE land cover labels for the tile
We will fetch CORINE Land Cover (100 m) and upsample it to the Sentinel-2 10 m grid for the chosen tile. We keep nearest-neighbor resampling so class IDs stay intact.

In [ ]:
# Load CORINE 2018 labels (Europe + Iceland) and align to Sentinel-2 grid
corine_year = "2018"
corine_raw = ee.Image('COPERNICUS/CORINE/V20/100m/2018').select('landcover')


# Match Sentinel-2 projection (10 m) while keeping nearest-neighbor labels
reference_proj = ee.Image(tile_collection.first()).select('B2').projection()
corine_aligned = corine_raw.resample('nearest').reproject(crs=reference_proj, scale=10)


# Optional: a small legend snippet for quick reference (full legend in CORINE docs)
corine_label_map = {
    111: "Continuous urban fabric",
    112: "Discontinuous urban fabric",
    121: "Industrial/commercial",
    211: "Non-irrigated arable land",
    231: "Pastures",
    311: "Broad-leaved forest",
    312: "Coniferous forest",
    321: "Natural grasslands",
    324: "Transitional woodland-shrub",
    332: "Bare rocks",
    512: "Water bodies"
}


print(f"Loaded CORINE {corine_year} layer and aligned it to the Sentinel-2 grid for tile {mgrs_tile}.")

### Export Complete Tiles

Export each of the 4 selected tiles as complete GeoTIFFs (no patching yet):

In [ ]:
# Export to Google Drive (full tile + label)
def export_tile_to_drive(image, date_str, tile_num):
    """Export single Sentinel-2 tile to Google Drive"""
    task = ee.batch.Export.image.toDrive(
        image=image,
        description=f'S2_{mgrs_tile}_Tile_{tile_num}_{date_str}',
        folder='iceland_ml_course',
        fileNamePrefix=f'sentinel2_{mgrs_tile}_tile{tile_num}_{date_str}',
        region=tile_geom,
        scale=10,  # 10 m resolution
        maxPixels=1e13,  # Large tiles need larger maxPixels
        fileFormat='GeoTIFF'
    )
    task.start()
    return task


def export_corine_to_drive():
    task = ee.batch.Export.image.toDrive(
        image=corine_aligned,
        description=f'CORINE_{mgrs_tile}_{corine_year}',
        folder='iceland_ml_course',
        fileNamePrefix=f'corine_{mgrs_tile}_{corine_year}',
        region=tile_geom,
        scale=10,
        maxPixels=1e13,
        fileFormat='GeoTIFF'
    )
    task.start()
    return task


# Export each selected tile
print("🚀 Starting tile export tasks...\n")
tasks = []
for i, date in enumerate(selected_dates, 1):
    img = ee.Image(masked_collection.toList(num_scenes).get(i-1))
    task = export_tile_to_drive(img, date.replace("-", ""), i)
    tasks.append(task)
    print(f"   ✓ Tile {i} export started: {date}")
    print(f"      Size: ~100-150 MB (complete Sentinel-2 scene)")


# Export CORINE once per tile
corine_task = export_corine_to_drive()
print(f"   ✓ CORINE label export started for tile {mgrs_tile} ({corine_year})")


print(f"\n📌 Monitor progress: https://code.earthengine.google.com/tasks")
print(f"📂 Files appear in Google Drive: iceland_ml_course/")
print(f"💾 Download tiles to JURECA: $PROJECT_training2600/data/sentinel2/")

🚀 Starting export tasks...

   ✓ Task 1 started: 2024-06-29
   ✓ Task 2 started: 2024-09-13
   ✓ Task 3 started: 2024-09-15
   ✓ Task 4 started: 2024-07-07

📌 Monitor progress at: https://code.earthengine.google.com/tasks
📂 Files will appear in Google Drive: iceland_ml_course/


In [ ]:
# Method 2: Direct download for the tile (faster for this lab)
import requests
import os


# Save to project directory
output_dir = Path(os.getenv('SCRATCH_training2600')) / Path(os.getenv('USER')) / 'data' / 'sentinel2'
output_dir.mkdir(parents=True, exist_ok=True)


print(f"💾 Downloading scenes to: {output_dir}\n")


geemap.ee_export_image_collection(
    masked_collection,
    out_dir=output_dir,
    scale=10,
    region=tile_geom,
    file_per_band=False
)


# Download CORINE labels (once)
corine_path = output_dir / f"corine_{mgrs_tile}_{corine_year}.tif"
geemap.ee_export_image(
    corine_aligned,
    filename=str(corine_path),
    scale=10,
    region=tile_geom
)


print(f"\n✅ Download complete! CORINE saved to {corine_path}")

💾 Downloading scenes to: /p/scratch/training2600/hashim1/data/sentinel2

Total number of images: 4

Exporting 1/4: /p/scratch/training2600/hashim1/data/sentinel2/20240629T125309_20240629T125303_T27WVM.tif
Generating URL ...
Please wait ...
Data downloaded to /p/scratch/training2600/hashim1/data/sentinel2/20240629T125309_20240629T125303_T27WVM.tif


Exporting 2/4: /p/scratch/training2600/hashim1/data/sentinel2/20240913T131259_20240913T131255_T27WWM.tif
Generating URL ...
Please wait ...
Data downloaded to /p/scratch/training2600/hashim1/data/sentinel2/20240913T131259_20240913T131255_T27WWM.tif


Exporting 3/4: /p/scratch/training2600/hashim1/data/sentinel2/20240915T130301_20240915T130257_T27WVM.tif
Generating URL ...
Please wait ...
Data downloaded to /p/scratch/training2600/hashim1/data/sentinel2/20240915T130301_20240915T130257_T27WVM.tif


Exporting 4/4: /p/scratch/training2600/hashim1/data/sentinel2/20240707T130301_20240707T130301_T27WWM.tif
Generating URL ...
Please wait ...
Data do

### Save Scene Metadata

In [ ]:
### Save Tile Metadata & AOI Definition


# Ensure output directory exists (in case you only used Drive exports)
if 'output_dir' not in locals():
    output_dir = Path(os.getenv('SCRATCH_training2600')) / Path(os.getenv('USER')) / 'data' / 'sentinel2'
    output_dir.mkdir(parents=True, exist_ok=True)


# Save metadata for the selected tiles
metadata_file = output_dir / 'tile_metadata.csv'
tile_metadata = df_scenes_sorted.head(num_scenes).copy()
tile_metadata['mgrs_tile'] = mgrs_tile
tile_metadata['export_dir'] = str(output_dir)
tile_metadata['corine_label'] = str(output_dir / f"corine_{mgrs_tile}_{corine_year}.tif")
tile_metadata.to_csv(metadata_file, index=False)


print(f"📝 Saved tile metadata to: {metadata_file}")
print(f"\n{tile_metadata.to_string(index=False)}")


# Save AOI and tile definition for reference
aoi_file = output_dir / 'aoi_definition.json'
aoi_dict = {
    'type': 'polygon',
    'coordinates': aoi_coords,
    'description': 'User AOI used to choose Sentinel-2 tile',
    'mgrs_tile': mgrs_tile,
    'tile_area_km2': tile_area_km2,
    'aoi_area_km2': aoi_area_km2,
    'corine_year': corine_year
}
with open(aoi_file, 'w') as f:
    json.dump(aoi_dict, f, indent=2)


print(f"\n📍 Saved AOI definition to: {aoi_file}")
print(f"\n✅ Lab 3 complete! 4 tile acquisitions + CORINE label ready for preprocessing and fine-tuning")

📝 Saved metadata to: /p/scratch/training2600/hashim1/data/sentinel2/scene_metadata.csv

 cloud_cover       date                                                     scene_id  solar_azimuth  solar_zenith
    0.057001 2024-06-29 S2B_MSIL2A_20240629T125309_N0510_R138_T27WVM_20240629T135617     167.064513     41.703204
    0.432559 2024-09-13 S2B_MSIL2A_20240913T131259_N0511_R081_T27WWM_20240913T152106     179.630060     60.958178
    0.846198 2024-09-15 S2A_MSIL2A_20240915T130301_N0511_R038_T27WVM_20240915T155047     174.674799     61.819436
    1.300875 2024-07-07 S2A_MSIL2A_20240707T130301_N0510_R038_T27WWM_20240707T155249     172.913902     42.074995


## Summary & Next Steps


### What We Covered
✅ Set up Google Earth Engine authentication  
✅ Defined an AOI (user-chosen) within CORINE coverage  
✅ Identified the Sentinel-2 MGRS tile covering the AOI  
✅ Queried Sentinel-2 imagery with cloud filtering  
✅ Selected up to 4 clearest full-tile acquisitions  
✅ Exported complete tiles **and** matching CORINE labels


### Data Acquired
- **Tiles:** Up to 4 complete Sentinel-2 scenes for one MGRS tile
- **Labels:** CORINE land cover (2018) resampled to 10 m on the tile grid
- **Bands:** All 13 Sentinel-2 bands (B2-B12, B8A)
- **Resolution:** 10 m (imagery) + 10 m resampled labels
- **Format:** GeoTIFF
- **Cloud Cover:** < 20%
- **Size:** ~100-150 MB per tile (imagery) + single CORINE label file


### Data Structure
Each tile contains:
- Sentinel-2 bands at 10 m (20 m bands resampled by GEE)
- A separate CORINE label raster aligned to the same grid
- Metadata CSV linking dates, tile ID, and label path


### Key GEE Concepts
- **ImageCollection:** Time series of satellite images
- **Filtering:** Spatial (AOI), temporal (date range), cloud filters, and tile ID
- **Cloud Masking:** Remove cloudy pixels using SCL band
- **Export:** Server-side processing for large raster datasets


### Prepare for Lab 4
Next lab: **Data Preprocessing & Patch Extraction**
- Load the downloaded Sentinel-2 tiles and CORINE label
- Extract 224×224 patches
- Apply band selection and normalization
- Align patches with land cover labels for fine-tuning

---

## 🔵 OPTIONAL: Advanced Topics (For Early Finishers)

## 🔵 Advanced Topic 1: Spectral Indices

### Why Calculate Indices?
Spectral indices highlight specific features:
- Vegetation health (NDVI)
- Water bodies (NDWI)
- Burn severity (NBR)

### Common Indices

In [ ]:
# Calculate NDVI (Normalized Difference Vegetation Index)
# NDVI = (NIR - Red) / (NIR + Red)
def add_ndvi(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return image.addBands(ndvi)

# Calculate NDWI (Normalized Difference Water Index)
# NDWI = (Green - NIR) / (Green + NIR)
def add_ndwi(image):
    ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')
    return image.addBands(ndwi)

# Calculate NBR (Normalized Burn Ratio)
# NBR = (NIR - SWIR2) / (NIR + SWIR2)
def add_nbr(image):
    nbr = image.normalizedDifference(['B8', 'B12']).rename('NBR')
    return image.addBands(nbr)

# Apply to collection
collection_with_indices = collection.map(add_ndvi).map(add_ndwi).map(add_nbr)

# Get one image and visualize NDVI
image_ndvi = collection_with_indices.first()

# Visualization parameters for NDVI
ndvi_viz = {
    'bands': ['NDVI'],
    'min': -1,
    'max': 1,
    'palette': ['brown', 'yellow', 'green', 'darkgreen']
}

# Display (if using geemap)
# Map.addLayer(image_ndvi, ndvi_viz, 'NDVI')

print("Spectral indices calculated and added to images")

## 🔵 Advanced Topic 2: Cloud Masking with QA Bands

### QA60 Band
Sentinel-2 includes a QA60 band for cloud detection:
- Bit 10: Opaque clouds
- Bit 11: Cirrus clouds

In [ ]:
# Cloud masking function using QA60
def mask_s2_clouds(image):
    qa = image.select('QA60')
    
    # Bits 10 and 11 are clouds and cirrus
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    
    # Both flags should be zero (no clouds)
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(
           qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    
    return image.updateMask(mask)

# Apply cloud mask to collection
collection_masked = collection.map(mask_s2_clouds)

# Compare before and after
image_original = collection.first()
image_masked = collection_masked.first()

print("Cloud masking applied")
print("Original image bands:", image_original.bandNames().getInfo())
print("Masked image bands:", image_masked.bandNames().getInfo())

## 🔵 Advanced Topic 3: Temporal Analysis

### Time Series Analysis
Analyze how the landscape changes over time

In [ ]:
# Query images from different seasons
winter_collection = ee.ImageCollection('COPERNICUS/S2_SR') \
    .filterBounds(aoi) \
    .filterDate('2023-12-01', '2024-02-28') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))

summer_collection = ee.ImageCollection('COPERNICUS/S2_SR') \
    .filterBounds(aoi) \
    .filterDate('2023-06-01', '2023-08-31') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))

# Create median composites
winter_median = winter_collection.median()
summer_median = summer_collection.median()

print(f"Winter images: {winter_collection.size().getInfo()}")
print(f"Summer images: {summer_collection.size().getInfo()}")

# Calculate NDVI difference
winter_ndvi = winter_median.normalizedDifference(['B8', 'B4'])
summer_ndvi = summer_median.normalizedDifference(['B8', 'B4'])
ndvi_diff = summer_ndvi.subtract(winter_ndvi)

print("Seasonal comparison completed")

## 🔵 Advanced Topic 4: Multiple AOIs

### Compare Different Regions

In [ ]:
# Define multiple AOIs in Iceland
aois = {
    'thingvellir': ee.Geometry.Rectangle([-21.2, 64.2, -21.0, 64.3]),
    'reykjavik': ee.Geometry.Rectangle([-22.0, 64.1, -21.8, 64.2]),
    'vatnajokull': ee.Geometry.Rectangle([-16.8, 64.3, -16.3, 64.5])
}

# Query each AOI
for name, aoi_geom in aois.items():
    aoi_collection = ee.ImageCollection('COPERNICUS/S2_SR') \
        .filterBounds(aoi_geom) \
        .filterDate('2023-06-01', '2023-09-30') \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
    
    count = aoi_collection.size().getInfo()
    print(f"{name}: {count} images available")
    
    if count > 0:
        # Get median composite
        median_image = aoi_collection.median()
        print(f"  Median composite created for {name}")

## 🔵 Advanced Topic 5: Batch Export

### Export Multiple Scenes Automatically

In [ ]:
# Export all images in collection (up to 10)
image_list = collection.toList(10)  # Limit to 10 images
count = min(collection.size().getInfo(), 10)

for i in range(count):
    image = ee.Image(image_list.get(i))
    
    # Get image ID and date
    image_id = image.get('system:index').getInfo()
    date = ee.Date(image.get('system:time_start')).format('YYYY-MM-dd').getInfo()
    
    # Export task
    task = ee.batch.Export.image.toDrive(
        image=image.select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12']),
        description=f'S2_{date}_{i}',
        folder='GEE_Exports',
        fileNamePrefix=f'iceland_s2_{date}',
        region=aoi,
        scale=10,
        maxPixels=1e13,
        fileFormat='GeoTIFF'
    )
    
    task.start()
    print(f"Export task {i+1} started: {date}")

print(f"\n{count} export tasks submitted to Google Drive")

---


## ✅ Lab 3 Completion Checklist


### Core Tasks (Must Complete)
- [ ] GEE account authenticated
- [ ] AOI defined within CORINE coverage
- [ ] Sentinel-2 collection queried and filtered by date/cloud
- [ ] MGRS tile identified for the AOI
- [ ] Up to 4 scenes exported for that tile (Drive or direct download)
- [ ] CORINE label exported for the tile
- [ ] Metadata saved (tile_metadata.csv + aoi_definition.json)


### Optional Tasks (If Time Permits)
- [ ] Calculated spectral indices (NDVI, NDWI, NBR)
- [ ] Applied advanced cloud masking
- [ ] Created seasonal composites
- [ ] Compared multiple AOIs
- [ ] Set up batch export


---


## 📝 Homework / Async Learning
- Download any remaining exported scenes from Google Drive
- Transfer scenes and CORINE label to JURECA: `$PROJECT_training2600/data/sentinel2/`
- Explore GEE Code Editor: https://code.earthengine.google.com
- Try different AOIs and date ranges (still inside CORINE)


## 🚀 Next Lab Preview
**Lab 4: Data Preprocessing**
- Load Sentinel-2 GeoTIFFs with rasterio
- Load CORINE labels and align to imagery
- Apply normalization techniques
- Create train/val/test splits